In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

token = userdata.get('GH_TOKEN')          # saca el token de los Secretos
![ -d /content/yuca_transfer ] || git clone https://{token}@github.com/Frank-Dev007/yuca_transfer.git /content/yuca_transfer
%cd /content/yuca_transfer
!git pull
!git config user.name "Frank García"
!git config user.email "tu-correo-de-github@ejemplo.com"

!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
Cloning into '/content/yuca_transfer'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/yuca_transfer
Already up to date.
Python 3.13.16
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
!mkdir -p scripts src notebooks checkpoints docs/evidencias results/E0 results/E1 results/E2 results/E3 results/EXTRA
!touch src/__init__.py checkpoints/.gitkeep results/.gitkeep docs/evidencias/.gitkeep
!ls

checkpoints  docs  notebooks  README.md  results  scripts  src


In [3]:
%%writefile .gitignore
data/
__pycache__/
.ipynb_checkpoints/

Writing .gitignore


In [4]:
%%writefile scripts/preparar_datos.py
import shutil, zipfile
from pathlib import Path

ZIP_DRIVE = Path("/content/drive/MyDrive/yuca_datos/cassavaleafdata.zip")
DATA_DIR = Path("data")
ZIP_LOCAL = DATA_DIR / ZIP_DRIVE.name

DATA_DIR.mkdir(exist_ok=True)
if not ZIP_LOCAL.exists():
    print("Copiando desde Drive...")
    shutil.copy(ZIP_DRIVE, ZIP_LOCAL)
print("Tamaño del zip (MB):", round(ZIP_LOCAL.stat().st_size / 1e6, 1))

with zipfile.ZipFile(ZIP_LOCAL) as z:
    z.extractall(DATA_DIR)
print("Contenido de data/:", [p.name for p in DATA_DIR.iterdir()])

Writing scripts/preparar_datos.py


In [5]:
%%writefile scripts/verificar_datos.py
from pathlib import Path
from collections import Counter

ROOT = Path("data/cassavaleafdata")
ESPERADO = {"train": 5656, "validation": 1889, "test": 1885}   # cifras oficiales
EXT = {".jpg", ".jpeg", ".png"}

for split, esperado in ESPERADO.items():
    conteo = Counter()
    for clase in sorted(p for p in (ROOT / split).iterdir() if p.is_dir()):
        conteo[clase.name] = sum(1 for f in clase.iterdir() if f.suffix.lower() in EXT)
    total = sum(conteo.values())
    estado = "OK" if total == esperado else "REVISAR"
    print(f"\n{split}: {total} imágenes (esperado {esperado}) -> {estado}")
    for c, n in conteo.items():
        print(f"   {c:10s} {n:5d}  ({n / total:.1%})")

Writing scripts/verificar_datos.py


In [6]:
%%writefile src/utils.py
import random
import numpy as np
import torch

def fijar_semilla(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def obtener_dispositivo():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

Writing src/utils.py


In [7]:
!python scripts/preparar_datos.py
!python scripts/verificar_datos.py
!python scripts/verificar_datos.py > docs/evidencias/02_conteo_clases.txt

Copiando desde Drive...
Tamaño del zip (MB): 1354.1
Contenido de data/: ['cassavaleafdata', 'cassavaleafdata.zip']

train: 5656 imágenes (esperado 5656) -> OK
   cbb          466  (8.2%)
   cbsd        1443  (25.5%)
   cgm          773  (13.7%)
   cmd         2658  (47.0%)
   healthy      316  (5.6%)

validation: 1889 imágenes (esperado 1889) -> OK
   cbb          156  (8.3%)
   cbsd         482  (25.5%)
   cgm          258  (13.7%)
   cmd          887  (47.0%)
   healthy      106  (5.6%)

test: 1885 imágenes (esperado 1885) -> OK
   cbb          155  (8.2%)
   cbsd         481  (25.5%)
   cgm          258  (13.7%)
   cmd          886  (47.0%)
   healthy      105  (5.6%)


In [8]:
%%writefile docs/dataset.md
# Ficha del conjunto de datos

- **Nombre:** cassava (iCassava 2019), TensorFlow Datasets
- **Ficha oficial:** https://www.tensorflow.org/datasets/catalog/cassava (consultada el 2026-11-12)
- **Archivo:** https://storage.googleapis.com/emcassavadata/cassavaleafdata.zip (descargado el 2026-11-12)
- **Carga en Colab:** zip subido manualmente a Google Drive (MyDrive/yuca_datos) y copiado a la sesión con scripts/preparar_datos.py
- **Origen (homepage):** https://www.kaggle.com/c/cassava-disease/overview — competencia iCassava 2019
- **Versión:** 0.1.0
- **Licencia:** no especificada en la ficha de TFDS (la licencia CC BY 4.0 del pie de página corresponde al contenido del sitio web, no a las imágenes). Pendiente: verificar en la página de la competencia.
- **Clases (5):** cbb (plaga bacteriana), cbsd (raya marrón), cgm (ácaro verde), cmd (mosaico), healthy (sana)
- **Particiones oficiales:** train 5.656 · validation 1.889 · test 1.885 · total 9.430

## Distribución verificada (scripts/verificar_datos.py)

| Clase   | Train        | Validation  | Test        |
|---------|--------------|-------------|-------------|
| cbb     | 466 (8,2 %)  | 156 (8,3 %) | 155 (8,2 %) |
| cbsd    | 1443 (25,5 %)| 482 (25,5 %)| 481 (25,5 %)|
| cgm     | 773 (13,7 %) | 258 (13,7 %)| 258 (13,7 %)|
| cmd     | 2658 (47,0 %)| 887 (47,0 %)| 886 (47,0 %)|
| healthy | 316 (5,6 %)  | 106 (5,6 %) | 105 (5,6 %) |
| **Total** | **5656**   | **1889**    | **1885**    |

- **Formato:** imágenes RGB de tamaño variable.
- **Resolución:** (se medirá en el EDA)
- **Procedencia:** fotografías de campo; país y condiciones por verificar en el artículo de referencia.
- **Criterios de exclusión:** (definir tras el EDA)
- **Duplicados encontrados:** (se completará en el EDA)

## Referencia
Mwebaze, E., Gebru, T., Frome, A., Nsumba, S., & Tusubira, J. (2019).
iCassava 2019 Fine-Grained Visual Categorization Challenge. arXiv:1908.02900.

Writing docs/dataset.md


In [9]:
%%writefile docs/bitacora.md
# Bitácora del proyecto

## 2026-11-12 — Frank

### 1. Entorno de ejecución
**Problema:** Elegir framework y dónde ejecutar.
**Alternativas:** PyTorch o Keras; PC local (RTX 3050 4 GB) o Google Colab (T4).
**Decisión:** PyTorch en Google Colab (GPU Tesla T4).
**Prueba:** Clonación del repositorio y nvidia-smi en Colab.
**Resultado:** Entorno funcionando.
**Revisión de la decisión:** (completar: por qué Colab y no el PC)

### 2. Acceso a GitHub desde Colab
**Problema:** Clonar y subir cambios a un repositorio privado desde Colab.
**Alternativas:** Repositorio público o token personal guardado como secreto de Colab.
**Decisión:** Token fine-grained (solo yuca_transfer, Contents: read/write) guardado como secreto GH_TOKEN.
**Prueba:** Clonación y push desde Colab.
**Resultado:** (completar)
**Revisión de la decisión:** (completar)

### 3. Obtención de los datos
**Problema:** Cargar el dataset en Colab.
**Alternativas:** Descarga automática por script o carga manual desde Google Drive.
**Decisión:** Carga manual del zip a Drive y copia a la sesión con scripts/preparar_datos.py.
**Prueba:** scripts/verificar_datos.py.
**Resultado:** (completar con lo que mostró la Celda 7)
**Revisión de la decisión:** (completar: motivo de preferir la carga manual)

### Consultas a IA
- Consulta: qué framework y entorno usar; cómo configurar Colab y GitHub; cómo cargar y verificar los datos.
- Respuesta recibida (resumen): sugerencia de PyTorch y Colab; código de configuración y verificación de datos.
- Decisión propia: (completar)

Writing docs/bitacora.md


In [10]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	checkpoints/
	docs/
	results/
	scripts/
	src/

nothing added to commit but untracked files present (use "git add" to track)


In [11]:
!git add .
!git commit -m "Estructura del proyecto, scripts de datos, verificación de particiones, ficha del dataset y bitácora"
!git push

[main 1068cf0] Estructura del proyecto, scripts de datos, verificación de particiones, ficha del dataset y bitácora
 11 files changed, 134 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 checkpoints/.gitkeep
 create mode 100644 docs/bitacora.md
 create mode 100644 docs/dataset.md
 create mode 100644 docs/evidencias/.gitkeep
 create mode 100644 docs/evidencias/02_conteo_clases.txt
 create mode 100644 results/.gitkeep
 create mode 100644 scripts/preparar_datos.py
 create mode 100644 scripts/verificar_datos.py
 create mode 100644 src/__init__.py
 create mode 100644 src/utils.py
Enumerating objects: 16, done.
Counting objects: 100% (16/16), done.
Delta compression using up to 2 threads
Compressing objects: 100% (12/12), done.
Writing objects: 100% (15/15), 3.81 KiB | 3.81 MiB/s, done.
Total 15 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/Frank-Dev007/yuca_transfer.git
   f368f0e..1068cf0  main -> main
